# 04 — Stability, trim and CG

VLM plus an empirical pod correction predicts neutral point and twist. Panknin
is an independent preliminary comparison, not a validation. Elevon trim uses
the chosen washout and real surface spans. Balance alternatives with the actual
CAD mass moment appear in notebook 07; a sizing-only run cannot establish CG.


## Reference frame, trim and sensitivity

**Inputs:** `planform`, section-model outputs, and study ranges. **Implementation:**
`uavdesign/aero/stability.py`. Aircraft coordinates are millimetres: $x$ aft of
the root leading edge, $y$ right, $z$ up. Angles in the tables are degrees.

Static margin is $SM=(x_{NP}-x_{CG})/\bar c$, where $\bar c$ is MAC. A positive
margin puts CG forward of the predicted neutral point. The model's moment
relation is $C_m(C_L,x_{CG})=C_{m,0}-SM C_L$; the washout solve brackets a root
of that expression at the target margin. The elevon solve uses actual span and
hinge fractions, and rejects a trim demand outside its solver bracket.

Plain-text equations: `static_margin = (neutral_point_x - CG_x) / MAC`;
`Cm_at_CG = Cm0 - static_margin * CL`; trim requires `Cm_at_CG = 0`.

VLM describes attached-flow loading; section viscous drag and stall do not
emerge from that solution. The empirical pod neutral-point correction introduces
additional uncertainty. Panknin is a different preliminary formulation but shares
section inputs, so agreement is not an independent experiment. The generated CAD
adds finite trailing-edge thickness and manufacturing detail to the aerodynamic
section representation.

**External review:** compare target CG with the manifest's realizable balance,
then assess trim at that CG, control authority, pitch damping and stall behavior.
A range with converged trim roots is not an approved flight envelope. Notebook 09
shows the battery-position lever arm without claiming a new trim qualification.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=True)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
from uavdesign.aero import stability as st, airfoil as af
display(pd.DataFrame([d.stability.__dict__]))
display(pd.DataFrame([{"quarter_mac_np_mm": d.wing.x_np_mm,
                      "vlm_wing_np_mm": d.stability.x_np_wing_mm,
                      "pod_shift_mm": d.stability.x_np_wing_mm-d.stability.x_np_mm,
                      "total_np_mm": d.stability.x_np_mm, "target_cg_mm": d.x_cg_target_mm}]))
cl = d.selected.summary()["cl_cruise"]
re = d.selected.polar_info["reynolds"]
pod = (pl["pod_length_mm"], pl["pod_width_mm"])
root_polar = af.section_polar(pl["airfoil_root"], re)
root_summary = af.summarize(pl["airfoil_root"], re, cl)
a0_root = float(np.interp(0, root_polar["cl"], root_polar["alpha_deg"]))
rows = []
for tip in studies["tip_candidates"]:
    twist, result = (d.washout_deg, d.stability) if tip == pl["airfoil_tip"] else st.solve_washout(
        d.wing, pl["airfoil_root"], tip, cl, pl["static_margin"], pod=pod,
        blend_start_mm=pl["pod_width_mm"]/2)
    tip_polar = af.section_polar(tip, re)
    tip_summary = af.summarize(tip, re, cl)
    a0_tip = float(np.interp(0, tip_polar["cl"], tip_polar["alpha_deg"]))
    p = st.panknin_washout_deg(root_summary.cm_at_cl_design, tip_summary.cm_at_cl_design,
                              a0_root, a0_tip, cl, pl["static_margin"], d.wing.aspect_ratio,
                              pl["taper"], d.wing.placed_panels[0].sweep_deg_at(0.25))
    rows.append({"tip": tip, "vlm_washout_deg": twist, "panknin_washout_deg": p,
                 "neutral_point_mm": result.x_np_mm, "trim_residual_cm": result.cm0-pl["static_margin"]*cl})
display(pd.DataFrame(rows))


In [ ]:
start, end = pl["elevon_span_fraction"]
model = st.WingAeroModel(pl["airfoil_root"], pl["airfoil_tip"], d.washout_deg,
                        st.ElevonSpec(start*d.wing.half_span_mm, end*d.wing.half_span_mm,
                                      1-pl["elevon_chord_fraction"]), pl["pod_width_mm"]/2)
rows = []
for margin in studies["static_margins"]:
    cg = d.stability.x_np_mm - margin*d.stability.mac_mm
    elevon, result = st.solve_elevon_trim(d.wing, model, cl, cg, pod=pod,
                                        airspeed_m_s=d.selected.cruise_speed_m_s)
    rows.append({"static_margin": margin, "cg_x_mm": cg, "elevon_deg": elevon,
                 "trim_residual_cm": result.cm_at(cl, cg)})
display(pd.DataFrame(rows))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r["static_margin"]*100 for r in rows], [r["elevon_deg"] for r in rows], "o-")
ax.set(xlabel="Static margin [% MAC]", ylabel="Elevon deflection [deg, negative up]"); ax.grid(alpha=.3)
save(fig, "04_trim_vs_static_margin.png")
print("Investigated CG interval [mm]:", min(r["cg_x_mm"] for r in rows), max(r["cg_x_mm"] for r in rows))
print("This is an investigated range, not a flight-approved CG envelope.")
